<a href="https://colab.research.google.com/github/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/blob/main/students/rawiya_mahveen/code/01_canada_real_estate_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canada Real Estate Map — Phase I / Phase II Prototype

**Goal:** Explore where homes are for sale and rent across all 10 Canadian provinces and 3 territories.

**Important:** Building footprints show building locations/shapes, not current sale or rental status. The property markers in this prototype are invented demonstration records, NOT real listings. A separate authorized listing source is needed before showing actual properties.

The Microsoft Canadian Building Footprints dataset contains approximately 11.8 million computer-generated building polygons across Canada and is licensed under ODbL. The files are large, so this notebook creates a national inventory and downloads only one region when explicitly requested.


In [52]:

import os
import subprocess
from getpass import getpass

GH_USER = "Rawiya-11"
GH_EMAIL = "rawiyamahveen11@gmail.com"
FOLDER = "students/rawiya_mahveen/"

OWNER = "AdityaR28"
REPO = "OSS4008A---Introduction-to-Remote-Sensing-and-Applications"

token = getpass("Enter your GitHub token: ")

os.chdir("/content")

repo_path = f"/content/{REPO}"

if not os.path.exists(repo_path):
    subprocess.run(
        ["git", "clone", f"https://github.com/{OWNER}/{REPO}.git"],
        check=True
    )

os.chdir(repo_path)

subprocess.run(
    ["git", "config", "--global", "credential.helper", "store"],
    check=True
)

credential = (
    "protocol=https\n"
    "host=github.com\n"
    f"username={GH_USER}\n"
    f"password={token}\n\n"
)

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

subprocess.run(["git", "config", "user.name", GH_USER], check=True)
subprocess.run(["git", "config", "user.email", GH_EMAIL], check=True)

subprocess.run(
    ["git", "remote", "set-url", "origin",
     f"https://github.com/{OWNER}/{REPO}.git"],
    check=True
)

subprocess.run(["git", "pull"], check=True)

print("Connected to the repository.")
print("Your current folder:", os.getcwd())


Enter your GitHub token: ··········
Connected to the repository.
Your current folder: /content/OSS4008A---Introduction-to-Remote-Sensing-and-Applications


In [60]:

student_path = f"students/{FOLDER}"

os.makedirs(f"{student_path}/data", exist_ok=True)
os.makedirs(f"{student_path}/code", exist_ok=True)
os.makedirs(f"{student_path}/results", exist_ok=True)

print("Your project folder:")
print(os.listdir(student_path))


Your project folder:
['data', 'code', 'results']


## 1. Install and import libraries

In [53]:
!pip -q install folium ipywidgets pandas requests

In [54]:
import os, zipfile, requests
from urllib.parse import quote
import pandas as pd
import folium
from folium.plugins import MarkerCluster, Fullscreen
import ipywidgets as widgets
from IPython.display import display, clear_output
print("Setup complete.")

Setup complete.


## 2. National building-footprint dataset inventory

In [55]:
BASE_URL = "https://minedbuildings.z5.web.core.windows.net/legacy/canadian-buildings-v2/"
building_counts = {
    "Alberta": 1777439, "British Columbia": 1359628, "Manitoba": 632982,
    "New Brunswick": 350989, "Newfoundland and Labrador": 255568,
    "Northwest Territories": 13161, "Nova Scotia": 402358, "Nunavut": 2875,
    "Ontario": 3781847, "Prince Edward Island": 76590, "Quebec": 2495801,
    "Saskatchewan": 681553, "Yukon": 11395
}
regions = list(building_counts.keys())
inventory = pd.DataFrame({
    "Region": regions,
    "Approx_Building_Count": [building_counts[r] for r in regions],
    "Download_URL": [BASE_URL + quote(r, safe="") + ".zip" for r in regions]
})
display(inventory)
print("Regions:", len(inventory))
print("Approximate national footprint count:", f"{inventory.Approx_Building_Count.sum():,}")

,Region,Approx_Building_Count,Download_URL
0,Alberta,1777439,https://minedbuildings.z5.web.core.windows.net...
1,British Columbia,1359628,https://minedbuildings.z5.web.core.windows.net...
2,Manitoba,632982,https://minedbuildings.z5.web.core.windows.net...
3,New Brunswick,350989,https://minedbuildings.z5.web.core.windows.net...
4,Newfoundland and Labrador,255568,https://minedbuildings.z5.web.core.windows.net...
5,Northwest Territories,13161,https://minedbuildings.z5.web.core.windows.net...
6,Nova Scotia,402358,https://minedbuildings.z5.web.core.windows.net...
7,Nunavut,2875,https://minedbuildings.z5.web.core.windows.net...
8,Ontario,3781847,https://minedbuildings.z5.web.core.windows.net...
9,Prince Edward Island,76590,https://minedbuildings.z5.web.core.windows.net...


Regions: 13
Approximate national footprint count: 11,842,186


## 3. Download one region only\nSome archives are hundreds of MB. Do not download every region into one Colab session or commit raw archives to GitHub.

In [56]:
def download_region(region, folder="/content/canada_building_footprints"):
    if region not in regions:
        raise ValueError("Choose a region from the national inventory.")
    os.makedirs(folder, exist_ok=True)
    url = BASE_URL + quote(region, safe="") + ".zip"
    zip_path = os.path.join(folder, region.replace(" ", "_") + ".zip")
    print("Downloading", region, "- this may take a while.")
    response = requests.get(url, stream=True, timeout=180)
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    extract_path = os.path.join(folder, region.replace(" ", "_"))
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_path)
    print("Extracted to:", extract_path)
    return extract_path

# Example: uncomment to download ONE region only:
# alberta_folder = download_region("Alberta")

## 4. Demonstration listing records\nThese invented records are only to test the UI.

In [57]:
import pandas as pd
import random

random.seed(42)

# Canadian cities and their approximate coordinates
cities = {
    "Vancouver": ("British Columbia", 49.2827, -123.1207),
    "Victoria": ("British Columbia", 48.4284, -123.3656),
    "Calgary": ("Alberta", 51.0447, -114.0719),
    "Edmonton": ("Alberta", 53.5461, -113.4938),
    "Regina": ("Saskatchewan", 50.4452, -104.6189),
    "Winnipeg": ("Manitoba", 49.8951, -97.1384),
    "Toronto": ("Ontario", 43.6532, -79.3832),
    "Ottawa": ("Ontario", 45.4215, -75.6972),
    "Montreal": ("Quebec", 45.5019, -73.5674),
    "Quebec City": ("Quebec", 46.8139, -71.2080),
    "Halifax": ("Nova Scotia", 44.6488, -63.5752),
    "Fredericton": ("New Brunswick", 45.9636, -66.6431),
    "Charlottetown": ("Prince Edward Island", 46.2382, -63.1311),
    "St. John's": ("Newfoundland and Labrador", 47.5615, -52.7126),
    "Whitehorse": ("Yukon", 60.7212, -135.0568),
    "Yellowknife": ("Northwest Territories", 62.4540, -114.3718),
    "Iqaluit": ("Nunavut", 63.7467, -68.5170)
}

property_types = [
    "Detached home",
    "Semi-detached home",
    "Townhouse",
    "Condominium",
    "Apartment"
]

# Approximate demonstration price ranges
sale_prices = {
    "Detached home": (350000, 1200000),
    "Semi-detached home": (300000, 850000),
    "Townhouse": (250000, 750000),
    "Condominium": (200000, 800000),
    "Apartment": (180000, 600000)
}

rent_prices = {
    "Detached home": (1600, 4000),
    "Semi-detached home": (1500, 3200),
    "Townhouse": (1400, 3000),
    "Condominium": (1400, 3200),
    "Apartment": (1000, 2800)
}

records = []

# Generate 10 demonstration properties per city
for city, (region, latitude, longitude) in cities.items():

    for i in range(10):

        listing_type = random.choice(["For Sale", "For Rent"])
        property_type = random.choice(property_types)

        if listing_type == "For Sale":
            price = random.randrange(
                sale_prices[property_type][0],
                sale_prices[property_type][1],
                5000
            )
        else:
            price = random.randrange(
                rent_prices[property_type][0],
                rent_prices[property_type][1],
                50
            )

        records.append({
            "City": city,
            "Region": region,
            "Listing_Type": listing_type,
            "Property_Type": property_type,
            "Price": price,
            "Latitude": latitude + random.uniform(-0.035, 0.035),
            "Longitude": longitude + random.uniform(-0.035, 0.035)
        })

# Create the new demonstration dataset
demo = pd.DataFrame(records)

print("Total demonstration properties:", len(demo))
print("Number of cities:", demo["City"].nunique())

display(demo.head(15))


Total demonstration properties: 170
Number of cities: 17


,City,Region,Listing_Type,Property_Type,Price,Latitude,Longitude
0,Vancouver,British Columbia,For Sale,Detached home,700000,49.264842,-123.145932
1,Vancouver,British Columbia,For Sale,Apartment,235000,49.289034,-123.153475
2,Vancouver,British Columbia,For Sale,Semi-detached home,445000,49.283075,-123.153842
3,Vancouver,British Columbia,For Sale,Apartment,445000,49.263131,-123.114451
4,Vancouver,British Columbia,For Sale,Semi-detached home,745000,49.277283,-123.136249
5,Vancouver,British Columbia,For Sale,Townhouse,315000,49.254192,-123.148930
6,Vancouver,British Columbia,For Rent,Apartment,1800,49.304199,-123.104619
7,Vancouver,British Columbia,For Sale,Condominium,250000,49.286343,-123.097642
8,Vancouver,British Columbia,For Rent,Apartment,1600,49.297020,-123.152492
9,Vancouver,British Columbia,For Sale,Townhouse,300000,49.307572,-123.095046


## 5. Interactive Canada map with filters

In [58]:

# Interactive Canada Real Estate Map

import folium
import ipywidgets as widgets

from IPython.display import display, clear_output
from folium.plugins import MarkerCluster, Fullscreen

# Create filter controls
region_filter = widgets.Dropdown(
    options=["All Canada"] + regions,
    description="Region:",
    layout=widgets.Layout(width="350px")
)

type_filter = widgets.Dropdown(
    options=["All", "For Sale", "For Rent"],
    description="Listing:",
    layout=widgets.Layout(width="250px")
)

price_filter = widgets.IntRangeSlider(
    value=[0, 1000000],
    min=0,
    max=1000000,
    step=25000,
    description="Price:",
    readout_format=",",
    layout=widgets.Layout(width="600px")
)

# Output areas
map_out = widgets.Output()
table_out = widgets.Output()


# Function to update the map
def render_map(change=None):

    # Keep Canadian records only
    data = demo[demo["Region"].isin(regions)].copy()

    # Apply region filter
    if region_filter.value != "All Canada":
        data = data[data["Region"] == region_filter.value]

    # Apply listing filter
    if type_filter.value != "All":
        data = data[data["Listing_Type"] == type_filter.value]

    # Apply price filter
    lo, hi = price_filter.value
    data = data[data["Price"].between(lo, hi)]

    # Create Canada-focused map
    m = folium.Map(
        location=[56.1304, -106.3468],
        zoom_start=3.5,
        tiles=None,
        control_scale=True,
        max_bounds=True
    )


    # Add background map
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles © Esri",
        name="Esri World Street Map",
        overlay=False,
        control=True
    ).add_to(m)

    # Fullscreen control
    Fullscreen(position="topright").add_to(m)

    # Group markers
    cluster = MarkerCluster(
        name="DEMO Canadian properties"
    ).add_to(m)

    # Add property markers
    for _, r in data.iterrows():

        popup = (
            f"<b>DEMO DATA — NOT A REAL LISTING</b><br>"
            f"City: {r.City}<br>"
            f"Region: {r.Region}<br>"
            f"Type: {r.Listing_Type}<br>"
            f"Property: {r.Property_Type}<br>"
            f"Demo price: ${r.Price:,.0f}"
        )

        folium.Marker(
            [r.Latitude, r.Longitude],
            tooltip=f"DEMO: {r.City} — {r.Listing_Type}",
            popup=folium.Popup(popup, max_width=280),
            icon=folium.Icon(
                color="green" if r.Listing_Type == "For Sale" else "blue",
                icon="home",
                prefix="fa"
            )
        ).add_to(cluster)

    # Layer control
    folium.LayerControl().add_to(m)

    # Display map
    with map_out:
        clear_output(wait=True)
        display(m)

    # Display property table
    with table_out:
        clear_output(wait=True)

        print("Canadian demonstration records shown:", len(data))

        display(
            data[
                ["City", "Region", "Listing_Type",
                 "Property_Type", "Price"]
            ].reset_index(drop=True)
        )


# Connect filters to map
for control in [region_filter, type_filter, price_filter]:
    control.observe(render_map, names="value")


# Display interface
display(widgets.VBox([
    widgets.HTML(
        "<h3>Canada Real Estate Map — Interactive Prototype</h3>"
        "<b>DEMONSTRATION DATA ONLY — not live listings</b>"
    ),
    widgets.HBox([region_filter, type_filter]),
    price_filter,
    map_out,
    table_out
]))

# Display initial map
render_map()


## 6. Export a standalone HTML map

In [59]:
overview = folium.Map(
    location=[56, -96],
    zoom_start=3,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map"
).add_to(overview)

cluster = MarkerCluster().add_to(overview)
for _, r in demo.iterrows():
    folium.Marker(
        [r.Latitude,r.Longitude],
        tooltip=f"DEMO ONLY: {r.City}",
        popup=f"DEMO DATA — NOT A REAL LISTING<br>{r.City}, {r.Region}<br>{r.Listing_Type}<br>${r.Price:,.0f}"
    ).add_to(cluster)
overview.save("/content/canada_real_estate_map_demo.html")
print("Saved: /content/canada_real_estate_map_demo.html")
from google.colab import files
files.download("/content/canada_real_estate_map_demo.html")

Saved: /content/canada_real_estate_map_demo.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>